# 模块R · R4 上机：系统文献综述（PRISMA 2020 方法论）

> **版本**：v5.0 学习材料包
> **配套**：notes.md（讲义）｜ data/README.md（真实数据/库）｜ solution.ipynb（参考答案）
> **聚焦**：PRISMA 2020 方法论本身（27条清单/偏倚评估/质量评价/ASReview主动学习机制）
> **与技能4 Day 1 区别**：技能4 Day 1 用PRISMA构建商业模式类型学；本单元理解PRISMA方法论原理


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> ⚠️ arxiv 包需要网络连接访问 arXiv API。pandas/scikit-learn/matplotlib 离线可用。
> 若无法联网，TODO1 的 solution 提供了 fallback 机制（基于真实 arXiv 历史查询的样本）。


In [ ]:
# !pip install arxiv pandas scikit-learn matplotlib -q
# arxiv 包需要网络连接；pandas/scikit-learn/matplotlib 离线可用


## 1. 数据背景与营销映射

**研究主题**：AI营销领域系统文献综述（PRISMA 2020 方法论实践）

**检索策略**（6条 arXiv 查询，PRISMA Phase 1 Identification）：

| 检索式 | arXiv 查询 | max_results | 用途 |
|--------|-----------|:-----------:|------|
| 检索式1 | `AI marketing` | 50 | AI营销核心 |
| 检索式2 | `LLM marketing` | 40 | LLM时代营销 |
| 检索式3 | `generative AI advertising` | 30 | 生成式AI广告 |
| 检索式4 | `AI content generation` | 30 | AI内容生成 |
| 检索式5 | `recommender system marketing` | 30 | 推荐系统营销 |
| 检索式6 | `AI consumer behavior` | 30 | AI消费者行为 |

**PRISMA 2020 四阶段对应**：

| 阶段 | TODO | 方法论学习点 |
|------|------|------------|
| Phase 1: Identification | TODO1-2 | 多查询交叉检索 + 去重 |
| Phase 2: Screening | TODO3 | 双盲筛选 + Cohen's kappa |
| Phase 3: Quality Assessment | TODO4 | Kitchenham五维 + RoB分级 |
| Phase 4: Synthesis | TODO5-6 | ASReview主动学习 + 流程图 |

**营销映射**：PRISMA综述"AI marketing"文献，但重点在方法论流程本身（非商业模式分类）。


## 2. 导入依赖

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## TODO 1：用 arxiv 包查询 arXiv API（PRISMA Phase 1: Identification）

**目标**：用 `arxiv` Python 包查询 arXiv API，获取6条"AI marketing"主题查询的论文元数据。

**PRISMA Item 5-6**（信息来源+检索式）：记录数据库名称、检索式、检索日期、结果数量。

**提示**：
- `import arxiv`
- `client = arxiv.Client(num_retries=3, page_size=50)`
- `search = arxiv.Search(query=q, max_results=mx, sort_by=arxiv.SortCriterion.Relevance)`
- 遍历 `client.results(search)` 获取每篇论文的 title/summary/published/primary_category/entry_id/authors
- 每条查询间 `time.sleep(3)` 避免 arXiv API 速率限制
- 若 API 不可用，加载 `data/arxiv_fallback.json`（真实查询存档）


In [ ]:
# 1. 用 arxiv 包查询 arXiv API，获取论文元数据

import arxiv

queries = [
    ("AI marketing", 50),
    ("LLM marketing", 40),
    ("generative AI advertising", 30),
    ("AI content generation", 30),
    ("recommender system marketing", 30),
    ("AI consumer behavior", 30),
]

papers = []
client = arxiv.Client(num_retries=3, page_size=50)

for q, mx in queries:
    try:
        search = arxiv.Search(query=q, max_results=mx, sort_by=arxiv.SortCriterion.Relevance)
        for r in client.results(search):
            papers.append({
                "title": r.title,
                "summary": r.summary[:400],
                "published": r.published.strftime("%Y-%m-%d"),
                "primary_category": r.primary_category,
                "entry_id": r.entry_id,
                "authors": [str(a) for a in r.authors][:3],
                "query": q,
            })
        print(f"  Query '{q}': fetched")
        time.sleep(3)  # arXiv API rate limit: 1 request per 3 seconds
    except Exception as e:
        print(f"  Query '{q}' ERROR: {e}")

# Fallback: if API unavailable, load archived real data
if len(papers) == 0 and os.path.exists("data/arxiv_fallback.json"):
    print("  API unavailable, loading fallback...")
    with open("data/arxiv_fallback.json") as f:
        fallback = json.load(f)
    papers = fallback["papers"]
    print(f"  Loaded {len(papers)} papers from fallback")

n_identified = len(papers)

print(f"\nPRISMA Phase 1 - Identification: {n_identified} papers identified")
print(f"Queries used: {len(queries)}")
if papers:
    print(f"Sample title: {papers[0]['title'][:80]}")


## TODO 2：PRISMA 去重（pandas）

**目标**：用 pandas 将论文列表转为 DataFrame，按标题去重，记录去重前后数量。

**PRISMA Item 16a**：记录识别阶段总数和去重后数量。

**提示**：
- `df = pd.DataFrame(papers)`
- 标题小写化后去重：`df['title_lower'] = df['title'].str.lower().str.strip()`
- `df_dedup = df.drop_duplicates(subset='title_lower')`
- 记录 `n_identified`（去重前）和 `n_after_dedup`（去重后）


In [ ]:
# 2. PRISMA 去重（pandas）

df = pd.DataFrame(papers)
n_identified = len(df)

# 按标题（小写化）去重
df['title_lower'] = df['title'].str.lower().str.strip()
df_dedup = df.drop_duplicates(subset='title_lower').reset_index(drop=True)
n_after_dedup = len(df_dedup)

# 提取年份
df_dedup['year'] = df_dedup['published'].str[:4].astype(int)

print(f"PRISMA Identification (before dedup): {n_identified}")
print(f"PRISMA After dedup: {n_after_dedup}")
print(f"Duplicates removed: {n_identified - n_after_dedup}")
print(f"Year range: {df_dedup['year'].min()}-{df_dedup['year'].max()}")


## TODO 3：PRISMA 双盲筛选 + Cohen's kappa（scikit-learn）

**目标**：用 pandas 执行 PRISMA 双盲筛选（年份+AI+营销相关性），用 scikit-learn 计算两位筛选者的 Cohen's kappa。

**PRISMA Item 7-8**（选择流程+筛选者）：记录筛选者数量、筛选标准、一致性。

**Cohen's kappa 等级**：
- < 0.20: 低（poor）
- 0.21-0.40: 一般（fair）
- 0.41-0.60: 中等（moderate）
- 0.61-0.80: 较好（substantial）
- 0.81-1.00: 优秀（almost perfect）

**提示**：
- 定义 AI 关键词和营销关键词列表
- 计算每篇论文的 relevance_score（AI关键词命中数 + 营销关键词命中数）
- ground_truth: relevance_score >= 4 且 year >= 2022
- 模拟第二位筛选者：在第一位基础上加 ~10% 噪声（翻转标签）
- `from sklearn.metrics import cohen_kappa_score`
- `kappa = cohen_kappa_score(rater1, rater2)`


In [ ]:
# 3. PRISMA 双盲筛选 + Cohen's kappa

from sklearn.metrics import cohen_kappa_score

ai_keywords = ['ai', 'artificial intelligence', 'llm', 'large language model', 'generative',
               'gpt', 'deep learning', 'machine learning', 'neural', 'transformer', 'bert', 'recommendation']
mkt_keywords = ['marketing', 'advertising', 'commerce', 'consumer', 'brand', 'content',
                'campaign', 'recommendation', 'personalization', 'customer', 'retail', 'product',
                'ad', 'engagement', 'conversion']

def relevance_score(row):
    text = (row['title'] + ' ' + row['summary']).lower()
    ai_count = sum(1 for k in ai_keywords if k in text)
    mkt_count = sum(1 for k in mkt_keywords if k in text)
    return ai_count + mkt_count

df_dedup['relevance_score'] = df_dedup.apply(relevance_score, axis=1)
df_dedup['ground_truth'] = ((df_dedup['relevance_score'] >= 4) & (df_dedup['year'] >= 2022)).astype(int)

# 模拟双盲筛选：第二位筛选者与第一位有~10%不一致
np.random.seed(42)
rater1 = df_dedup['ground_truth'].values.copy()
flip_mask = np.random.random(len(df_dedup)) < 0.10
rater2 = rater1.copy()
rater2[flip_mask] = 1 - rater2[flip_mask]

# Cohen's kappa
kappa = cohen_kappa_score(rater1, rater2)

# 双盲筛选：两位筛选者都认为相关的论文才纳入
df_dedup['rater1'] = rater1
df_dedup['rater2'] = rater2
n_screened = int((rater1 & rater2).sum())

print(f"PRISMA Phase 2 - Screening")
print(f"Ground truth positive (relevance>=4, year>=2022): {df_dedup['ground_truth'].sum()}")
print(f"Screened (dual rater agreement): {n_screened}")
print(f"Cohen's kappa: {kappa:.4f}")
kappa_level = "优秀" if kappa >= 0.81 else ("较好" if kappa >= 0.61 else ("中等" if kappa >= 0.41 else ("一般" if kappa >= 0.21 else "低")))
print(f"Agreement level: {kappa_level}")
print(f"Agreement: {(rater1 == rater2).sum()}/{len(rater1)} ({(rater1 == rater2).sum()/len(rater1)*100:.1f}%)")


## TODO 4：Kitchenham & Charters 五维质量评估 + Risk of Bias 分级

**目标**：实现 Kitchenham & Charters（2007）五维质量评估函数，对每篇论文打0-5分，按 Risk of Bias 三级分级。

**五维质量评估**（每维0-1分，总分0-5）：
1. 研究问题清晰度：标题/摘要是否明确陈述研究问题或目标？
2. 方法适当性：是否描述了研究方法/框架/算法？
3. 数据系统性：是否提及数据集/评估/实验？
4. 分析恰当性：是否报告了结果/发现/贡献？
5. 局限讨论：是否讨论了局限/挑战/未来工作？

**Risk of Bias 三级**：
- Low Risk: 质量分 >= 4
- Moderate Risk: 质量分 2-3
- High Risk: 质量分 0-1

**提示**：
- 定义 `quality_score(row)` 函数，检查标题+摘要中的关键词
- `df_dedup['quality_score'] = df_dedup.apply(quality_score, axis=1)`
- 定义 `rob_class(score)` 函数返回 Low/Moderate/High
- 筛选：screened论文中 quality_score >= 2 的纳入


In [ ]:
# 4. Kitchenham & Charters 五维质量评估 + RoB分级

def quality_score(row):
    # Kitchenham & Charters (2007) 五维质量评估，每维0-1分，总分0-5
    text = row['title'] + ' ' + row['summary']
    text_lower = text.lower()
    score = 0
    # Q1: 研究问题清晰度
    if '?' in text or 'we study' in text_lower or 'we investigate' in text_lower \
       or 'we propose' in text_lower or 'this paper' in text_lower:
        score += 1
    # Q2: 方法适当性
    if any(m in text_lower for m in ['method', 'approach', 'framework', 'model', 'algorithm', 'experiment']):
        score += 1
    # Q3: 数据系统性
    if any(d in text_lower for d in ['data', 'dataset', 'evaluation', 'results', 'experiment', 'benchmark']):
        score += 1
    # Q4: 分析恰当性
    if any(f in text_lower for f in ['we find', 'results show', 'contribution', 'outperform', 'improve', 'novel', 'achiev']):
        score += 1
    # Q5: 局限讨论
    if any(l in text_lower for l in ['limitation', 'future work', 'however', 'challenge', 'constraint']):
        score += 1
    return score

def rob_class(score):
    # Risk of Bias 三级分级
    if score >= 4:
        return 'Low'
    elif score >= 2:
        return 'Moderate'
    else:
        return 'High'

df_dedup['quality_score'] = df_dedup.apply(quality_score, axis=1)
df_dedup['rob'] = df_dedup['quality_score'].apply(rob_class)

# 纳入：双盲筛选通过（rater1 & rater2） + 质量分 >= 2
df_screened = df_dedup[(df_dedup['rater1'] == 1) & (df_dedup['rater2'] == 1)].copy()
df_included = df_screened[df_screened['quality_score'] >= 2].copy()
n_included = len(df_included)

print(f"PRISMA Phase 3 - Quality Assessment")
print(f"Quality score distribution:\n{df_dedup['quality_score'].value_counts().sort_index()}")
print(f"\nRisk of Bias distribution (included):\n{df_included['rob'].value_counts()}")
print(f"\nIncluded (quality>=2): {n_included}")
print(f"Mean quality score: {df_included['quality_score'].mean():.2f}")


## TODO 5：ASReview 主动学习筛选模拟（scikit-learn）

**目标**：用 scikit-learn 模拟 ASReview 的主动学习筛选机制，计算效率提升。

**ASReview 主动学习流程**：
1. **种子集标注**：人工标注少量论文（5篇正例 + 5篇负例）
2. **特征提取**：TF-IDF 将标题+摘要转为特征向量
3. **分类器训练**：LogisticRegression 训练
4. **迭代主动学习**：每轮查询排名最高的15篇，人工标注后加入训练集，重新训练
5. **效率计算**：按模型排名，读前N篇覆盖90%相关论文 vs 人工全筛

**提示**：
- `from sklearn.feature_extraction.text import TfidfVectorizer`
- `from sklearn.linear_model import LogisticRegression`
- `vectorizer = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1,2))`
- `X = vectorizer.fit_transform(texts)`
- 种子集：5个正例索引 + 5个负例索引
- 5轮迭代，每轮查询15篇
- 最终排序后计算 90% recall 需要读多少篇


In [ ]:
# 5. ASReview 主动学习筛选模拟

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

np.random.seed(42)

# 准备特征
X_text = (df_dedup['title'] + ' ' + df_dedup['summary']).values
y = df_dedup['ground_truth'].values
n_papers = len(df_dedup)

# TF-IDF 特征提取
vectorizer = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
X = vectorizer.fit_transform(X_text)

# 种子集：5个正例 + 5个负例
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
np.random.shuffle(pos_idx)
np.random.shuffle(neg_idx)

labeled = list(pos_idx[:5]) + list(neg_idx[:5])
unlabeled = [i for i in range(n_papers) if i not in labeled]

# 迭代主动学习：5轮，每轮查询排名最高的15篇
clf = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
for round_num in range(5):
    if len(unlabeled) == 0:
        break
    clf.fit(X[labeled], y[labeled])
    probs = clf.predict_proba(X[unlabeled])[:, 1]
    n_query = min(15, len(unlabeled))
    top_indices = np.argsort(probs)[::-1][:n_query]
    queried = [unlabeled[i] for i in top_indices]
    labeled.extend(queried)
    unlabeled = [i for i in unlabeled if i not in queried]

# 最终排序 + 效率计算
clf.fit(X[labeled], y[labeled])
all_probs = clf.predict_proba(X)[:, 1]
df_dedup['asreview_score'] = all_probs

ranked = df_dedup.sort_values('asreview_score', ascending=False).reset_index(drop=True)
ranked['cum_relevant'] = ranked['ground_truth'].cumsum()
total_relevant = int(ranked['ground_truth'].sum())

# 90% recall: 读前N篇覆盖90%的相关论文
threshold_90 = max(1, int(np.ceil(total_relevant * 0.9)))
papers_to_read_90 = int((ranked['cum_relevant'] >= threshold_90).idxmax() + 1)
efficiency = (1 - papers_to_read_90 / n_papers) * 100
speedup = n_papers / papers_to_read_90

print(f"=== ASReview Active Learning Simulation ===")
print(f"Total papers: {n_papers}")
print(f"Relevant papers (ground truth): {total_relevant}")
print(f"Seed set: 10, AL rounds: 5x15=75 queries, Total labeled: {len(labeled)}")
print(f"Papers to read (ASReview, 90% recall): {papers_to_read_90}")
print(f"Papers to read (manual, 100%): {n_papers}")
print(f"Efficiency: {efficiency:.1f}% reduction in screening effort")
print(f"Speedup: {speedup:.1f}x")
print(f"\nNote: Production ASReview achieves ~10x speedup via better features + query strategies")


## TODO 6：PRISMA 流程图 + Risk of Bias 汇总（matplotlib）

**目标**：用 matplotlib 画 PRISMA 2020 流程图（标注各阶段真实数字）+ Risk of Bias 汇总柱状图。

**PRISMA 流程图结构**：
```
[Identification: n_identified]
         |
[After dedup: n_after_dedup]
         |
[Screened: n_screened] --- excluded: n_after_dedup - n_screened
         |
[Included: n_included]  --- excluded: n_screened - n_included
```

**提示**：
- `from matplotlib.patches import FancyBboxPatch, FancyArrowPatch`
- 用 `FancyBboxPatch` 画方框
- 用 `FancyArrowPatch` 画箭头
- 右侧画排除数量的侧框
- 第二张子图画 Risk of Bias 分布柱状图


In [ ]:
# 6. PRISMA 流程图 + Risk of Bias 汇总

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: PRISMA Flow Diagram ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('PRISMA 2020 Flow Diagram\n(AI Marketing Systematic Review)', fontsize=13, fontweight='bold')

stages = [
    (2, 10, 6, 1.2, f'Identification\nn = {n_identified}', '#4ECDC4'),
    (2, 7.5, 6, 1.2, f'After Deduplication\nn = {n_after_dedup}', '#45B7D1'),
    (2, 5, 6, 1.2, f'Screened (dual rater)\nn = {n_screened}', '#96CEB4'),
    (2, 2.5, 6, 1.2, f'Included (quality >= 2)\nn = {n_included}', '#FFEAA7'),
]

for x, y, w, h, label, color in stages:
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Arrows between stages
for y_start in [10, 7.5, 5]:
    arrow = FancyArrowPatch((5, y_start), (5, y_start - 1.3),
                            arrowstyle='->', mutation_scale=20, color='black', linewidth=2)
    ax.add_patch(arrow)

# Exclusion side boxes
excluded_dedup = n_identified - n_after_dedup
excluded_screen = n_after_dedup - n_screened
excluded_quality = n_screened - n_included

side_info = [
    (8.5, 7.5, f'Duplicates\nremoved: {excluded_dedup}'),
    (8.5, 5, f'Excluded by\nscreening: {excluded_screen}'),
    (8.5, 2.5, f'Excluded by\nquality: {excluded_quality}'),
]
for x, y, label in side_info:
    ax.text(x, y + 0.6, label, ha='center', va='center', fontsize=9, style='italic', color='#666')
    arrow = FancyArrowPatch((8, y + 0.6), (8.3, y + 0.6),
                            arrowstyle='->', mutation_scale=15, color='gray', linewidth=1)
    ax.add_patch(arrow)

# Cohen's kappa annotation
ax.text(5, 0.8, f"Cohen's kappa = {kappa:.4f} (substantial agreement)",
        ha='center', va='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.8))

# --- Right: Risk of Bias Summary ---
ax2 = axes[1]
rob_counts = df_included['rob'].value_counts()
rob_order = ['Low', 'Moderate', 'High']
rob_vals = [rob_counts.get(r, 0) for r in rob_order]
colors_rob = ['#2ECC71', '#F39C12', '#E74C3C']

bars = ax2.bar(rob_order, rob_vals, color=colors_rob, edgecolor='black', linewidth=1.2)
ax2.set_title('Risk of Bias Summary\n(Kitchenham & Charters Quality Assessment)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Number of Papers', fontsize=11)
ax2.set_xlabel('Risk of Bias Level', fontsize=11)

for bar, val in zip(bars, rob_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)

# Quality score distribution as text
qs_dist = df_included['quality_score'].value_counts().sort_index()
ax2.text(0.5, 0.95, f"Quality scores: {dict(qs_dist)}",
         transform=ax2.transAxes, ha='center', va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('prisma_flow_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"PRISMA Flow Diagram saved.")
print(f"  Identification: {n_identified}")
print(f"  After dedup: {n_after_dedup}")
print(f"  Screened: {n_screened}")
print(f"  Included: {n_included}")


## 3. 2026前沿：ASReview + DeepSeek/RAGAS + 天道推演 + MCP

### ASReview：AI辅助系统性文献综述
ASReview（Utrecht University 开发）用**主动学习**加速PRISMA筛选阶段。本单元的TODO5模拟了其核心机制。生产级ASReview通过更优的特征提取+查询策略（uncertainty sampling / query-by-committee）可达 **10x加速**。

### DeepSeek/RAGAS：LLM辅助证据合成
- **DeepSeek-V3/R1**：开源LLM，成本仅GPT-4的1/10，可用于论文摘要提取和相关性语义判断
- **RAGAS**：评估LLM生成的综述文本质量（faithfulness / answer_relevancy / context_precision）
- LLM辅助文献综述是L1（关联分析），不能替代人工全文复筛（L2干预）

### 天道推演 x 研究空白预判
用天道推演+**贝叶斯推断**预判AI营销文献的研究空白演化：
- 沙盘分支1：AI营销Agent自主决策（Agent可靠性突破 -> 研究空白向Agent治理迁移）
- 沙盘分支2：LLM营销内容合规（监管收紧 -> 合规性研究爆发）
- 沙盘分支3：多模态营销智能（视觉+语言融合 -> 跨模态评估空白）

### MCP（Model Context Protocol）
MCP协议标准化LLM与外部工具连接。文献综述场景：MCP + arxiv自动化Phase 1检索，MCP + ASReview自动化Phase 2筛选，**多Agent仿真**协作完成全流程。

> 🔗 深入阅读见 [`reading.md`](./reading.md)


## 完成检查

完成以上6个TODO后，你应该能：
- [ ] 真实查询arXiv API获取论文元数据（PRISMA Phase 1）
- [ ] 用pandas执行PRISMA去重/筛选（Phase 1-2）
- [ ] 计算Cohen's kappa评分者一致性（PRISMA Item 7）
- [ ] 实现Kitchenham五维质量评估+RoB分级（Phase 3）
- [ ] 模拟ASReview主动学习筛选机制（Phase 4 Synthesis）
- [ ] 画PRISMA 2020流程图+偏倚风险汇总图

**方法论学习要点**（与技能4 Day 1的区别）：
- 技能4 Day 1：用PRISMA做工具 -> 构建商业模式类型学
- 本单元R4：理解PRISMA方法论本身 -> 27条清单/偏倚评估/质量评价/ASReview机制
